In [1]:
## Step 1: Raw image Pre-processing (Jupyter Notebook)

import numpy as np
import tifffile
import zipfile
import re
from pathlib import Path
from read_roi import read_roi_zip
from roifile import ImagejRoi
from skimage.draw import polygon

# --- 1. HELPER FUNCTIONS ---
def create_labels_and_map(rois, shape):
    labels = np.zeros(shape, dtype=np.int32)
    roi_map = {}
    label_id = 1
    for roi_name, roi_data in rois.items():
        if roi_data['type'] in ['polygon', 'freehand', 'traced']:
            x, y = np.array(roi_data['x']), np.array(roi_data['y'])
            rr, cc = polygon(y, x, shape)
            labels[rr, cc] = label_id
            roi_map[roi_name] = label_id
            label_id += 1
    return labels, roi_map

# --- 2. PARAMETERS ---
input_dir = Path(r"input_dir")
output_dir = Path(r"output_dir")

# OUTPUT FOLDERS
dir_merged = output_dir / "Merged_Crops"
dir_c1 = output_dir / "C1_Clover"
dir_c2 = output_dir / "C2_Ruby"
dir_rois = output_dir / "ROIs"
dir_rois_orig = dir_rois / "Original"
dir_rois_5x = dir_rois / "5x_Scaled"

for d in [dir_merged, dir_c1, dir_c2, dir_rois_orig, dir_rois_5x]:
    d.mkdir(parents=True, exist_ok=True)

PADDING = 10 
total_cells = 0
saved_isolated = 0
saved_overlapped = 0

# SAVE LIST WITH FILES WITH MORE THAN ONE CELL
overlapping_cells = []

# --- 3. PROCESSING ---
print(f"Starting pipeline in: {input_dir.name}\n")
image_files = list(input_dir.glob("*.tif")) + list(input_dir.glob("*.tiff"))

for image_path in image_files:
    # Match Zip file
    zip_name = image_path.stem.replace("_merged", "_rois_rois") + ".zip"
    zip_path = input_dir / zip_name
    if not zip_path.exists(): 
        print(f"  ❌ SKIPPED: File {zip_name} not found")
        continue
        
    print(f"Processing {image_path.name}...")
    image = tifffile.imread(image_path)
    rois = read_roi_zip(zip_path)
    
    if image.ndim == 3:
        num_channels, max_y_img, max_x_img = image.shape
    else:
        print(f"  ❌ SKIPPED: {image_path.name} does not have 3 dimensions.")
        continue
        
    labels, roi_map = create_labels_and_map(rois, (max_y_img, max_x_img))
    
    for roi_name, roi_data in rois.items():
        if roi_data['type'] not in ['polygon', 'freehand', 'traced']: continue
            
        total_cells += 1
        current_label_id = roi_map[roi_name]
        orig_x, orig_y = np.array(roi_data['x']), np.array(roi_data['y'])
        
        min_x, max_x = np.min(orig_x), np.max(orig_x)
        min_y, max_y = np.min(orig_y), np.max(orig_y)
        
        x_start, y_start = int(max(0, min_x - PADDING)), int(max(0, min_y - PADDING))
        x_end, y_end = int(min(max_x_img, max_x + PADDING)), int(min(max_y_img, max_y + PADDING))
        
        has_overlap = np.any((labels[y_start:y_end, x_start:x_end] != 0) & (labels[y_start:y_end, x_start:x_end] != current_label_id))
        
        # --- NAME CLEANING ---
        clean_name = image_path.stem.replace("_merged", "")
        roi_number = re.sub(r'\D', '', roi_name) 
        if not roi_number: roi_number = roi_name 
        
        base_crop_name = f"{clean_name}_{roi_number}"
        
        if has_overlap:
            overlapping_cells.append(base_crop_name)
            saved_overlapped += 1
        else:
            saved_isolated += 1
            
        cropped_merged = image[:, y_start:y_end, x_start:x_end]
        new_x, new_y = orig_x - x_start, orig_y - y_start
        
        # --- SAVE IMAGES ---
        tifffile.imwrite(dir_merged / f"{base_crop_name}_merged.tif", cropped_merged, imagej=True, metadata={'axes': 'CYX'})
        tifffile.imwrite(dir_c1 / f"{base_crop_name}_c1.tif", cropped_merged[0])
        tifffile.imwrite(dir_c2 / f"{base_crop_name}_c2.tif", cropped_merged[1])
            
        # --- SAVE ROIS ---
        def save_roi(coords, folder, suffix):
            roi = ImagejRoi.frompoints(coords)
            roi.name = roi_name
            temp_path = output_dir / "temp.roi"
            roi.tofile(temp_path)
            with zipfile.ZipFile(folder / f"{base_crop_name}_{suffix}.zip", 'w') as zf:
                zf.write(temp_path, arcname=f"{roi_name}.roi")
            if temp_path.exists():
                temp_path.unlink()

        save_roi(np.column_stack((new_x, new_y)), dir_rois_orig, "roi")
        save_roi(np.column_stack((new_x * 5, new_y * 5)), dir_rois_5x, "roi_5x")

# --- 4. SAVE LIST WITH OVERLAPPING CELLS ---
report_file = output_dir / "overlapping_cells.txt"
with open(report_file, "w", encoding="utf-8") as f:
    for item in overlapping_cells:
        f.write(f"{item}\n")

print("\n--- SUMMARY ---")
print(f"Total cells: {total_cells}")
print(f"Single-cell: {saved_isolated} | Cells with overlap: {saved_overlapped}")
print(f"Overlap file saved in: {report_file.name}")

Starting pipeline in: Merged

Processing 212_merged.tif...
Processing 113_merged.tif...
Processing 112_merged.tif...
Processing 215_merged.tif...
Processing 211_merged.tif...
Processing 210_merged.tif...
Processing 213_merged.tif...
Processing 214_merged.tif...
Processing 310_merged.tif...
Processing 311_merged.tif...
Processing 313_merged.tif...
Processing 518_merged.tif...
Processing 517_merged.tif...
Processing 514_merged.tif...
Processing 513_merged.tif...
Processing 512_merged.tif...
Processing 416_merged.tif...
Processing 418_merged.tif...
Processing 415_merged.tif...
Processing 414_merged.tif...
Processing 413_merged.tif...
Processing 412_merged.tif...
Processing 111_merged.tif...
Processing 110_merged.tif...
Processing 417_merged.tif...
Processing 510_merged.tif...
Processing 511_merged.tif...
Processing 515_merged.tif...
Processing 516_merged.tif...
Processing 611_merged.tif...
Processing 612_merged.tif...
Processing 613_merged.tif...
Processing 614_merged.tif...
Processing 61